# CodeBud Module 2: RAG code generator (Groq edition)
Pipeline:

`user request -> intent -> retrieve relevant snippets -> Groq LLM -> code`

- **Retriever:** local embeddings (`sentence-transformers`) + cosine similarity. No vector DB needed at this scale.
- **Generator:** an open model served by **Groq** (free tier available, rate-limited). Retrieved snippets go into the prompt so answers are grounded in *your* knowledge base.
- **Knowledge base:** any `.ipynb`, `.py`, `.md` or `.txt` files in a `knowledge_base/` folder next to this notebook.

In [4]:
# run once, then restart the kernel if needed
%pip install groq sentence-transformers numpy

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 8.3 MB/s  0:00:009.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 7.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 8.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 9.2 MB/s  0:00:00m 9.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 9.3 MB/s  0:00:01m 9.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 6.9 MB/s  0:00:007.2 MB/s eta 0:00:01
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ━

---
**1. API key and model**  
Create a free key at [console.groq.com](https://console.groq.com), then set it **before launching Jupyter**:
```bash
export GROQ_API_KEY='****'
```
Never paste the key into the notebook or commit it to GitHub. If the variable isn't set, the cell below asks for it securely.

Groq retires and adds models from time to time. If you get a "model not found" error, pick a current name from the [models page](https://console.groq.com/docs/models) and change `MODEL`.

In [40]:
import os
import getpass
from groq import Groq

key = getpass.getpass("Enter your Groq API key: ").strip()

client = Groq(api_key=key)

MODEL = 'openai/gpt-oss-120b'

Enter your Groq API key:  ········


In [41]:
print(client.models.list())

ModelListResponse(data=[Model(id='openai/gpt-oss-20b', created=1754407957, object='model', owned_by='OpenAI', model_card='https://openai.com/index/gpt-oss-model-card', active=True, context_window=131072, public_apps=None, max_completion_tokens=65536, hugging_face_id='openai/gpt-oss-20b', name='GPT OSS 20B', input_modalities=['text'], output_modalities=['text'], context_length=131072, max_output_length=65536, pricing={'prompt': '0.000000075', 'completion': '0.0000003', 'image': '0', 'request': '0', 'input_cache_read': '0.0000000375'}, supported_sampling_parameters=['temperature', 'top_p', 'stop', 'seed', 'max_tokens'], supported_features=['tools', 'json_mode', 'structured_outputs', 'reasoning']), Model(id='openai/gpt-oss-safeguard-20b', created=1761708789, object='model', owned_by='OpenAI', model_card='openai/gpt-oss-safeguard-20b', active=True, context_window=131072, public_apps=None, max_completion_tokens=65536, hugging_face_id='openai/gpt-oss-safeguard-20b', name='Safety GPT OSS 20B'

---
**2. Load and chunk the knowledge base**  
- `.py` files are split into one chunk per top-level function/class (so each chunk is a complete unit of code).  
- `.md` / `.txt` files are split into paragraph groups of about 800 characters.  
If `knowledge_base/` doesn't exist, a few sample snippets are created so you can test immediately. Replace them with your own material.

In [42]:

import json

def chunk_notebook(text, filename):
    notebook = json.loads(text)
    chunks = []

    for cell in notebook.get("cells", []):
        if cell["cell_type"] in ("code", "markdown"):
            content = "".join(cell.get("source", []))
            if content.strip():
                chunks.append({
                    "source": filename,
                    "text": content
                })

    return chunks


def load_knowledge_base(folder=KB_DIR):
    chunks = []

    for path in sorted(Path(folder).rglob("*")):
        if not path.is_file():
            continue

        text = path.read_text(encoding="utf-8", errors="ignore")

        if path.suffix == ".ipynb":
            chunks += chunk_notebook(text, path.name)
        elif path.suffix == ".py":
            chunks += chunk_python(text, path.name)
        elif path.suffix in (".md", ".txt"):
            chunks += chunk_text(text, path.name)

    return chunks


chunks = load_knowledge_base()
print(f"{len(chunks)} chunks loaded")


1135 chunks loaded


---
**3. Embed and retrieve**  
Each chunk becomes a vector. A query is embedded the same way, and we take the `k` chunks with the highest cosine similarity (vectors are normalised, so a dot product is enough). Retrieval runs fully locally and is free.

In [43]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')

def build_index(chunks):
    return embedder.encode([c['text'] for c in chunks], normalize_embeddings=True)

chunk_vecs = build_index(chunks)
print(chunk_vecs.shape)


def retrieve(query, k=3, min_score=0.2):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = chunk_vecs @ q
    top = np.argsort(-scores)[:k]
    return [(chunks[i], float(scores[i])) for i in top if scores[i] >= min_score]

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 6280.54it/s]


(1135, 384)


In [44]:
print("Number of chunks:", len(chunks))
print("chunk_vecs shape:", chunk_vecs.shape)
print("query embedding shape:", embedder.encode(["test"], normalize_embeddings=True)[0].shape)

Number of chunks: 1135
chunk_vecs shape: (1135, 384)
query embedding shape: (384,)


In [45]:
# sanity check retrieval before calling the API
for c, s in retrieve('how do I read a json file'):
    print(f'{s:.2f}  {c["source"]}')

0.23  q14.ipynb
0.23  naivebayes.ipynb
0.23  DecisionTree.ipynb


---
**4. Intent routing (hook for Module 1)**  
`classify_intent` is a **placeholder** using keywords. The Module 1 model was trained on SNIPS intents (PlayMusic, GetWeather, ...), which aren't coding intents, so it can't be dropped in directly. Options: retrain Module 1 on `write_code` / `explain_code` / `debug_code` examples, or keep this keyword router.

In [46]:
INTENT_INSTRUCTIONS = {
    'write_code':   'Write the requested code. Return a complete, runnable solution in a single code block, then a 1-2 sentence explanation.',
    'explain_code': 'Explain the code or concept clearly and concisely, using short examples where useful.',
    'debug_code':   'Find the bug, show the corrected code, and explain what was wrong in 1-3 sentences.',
}

def classify_intent(text):
    # TODO: replace with Module 1 prediction once it is trained on coding intents
    t = text.lower()
    if any(w in t for w in ('error', 'bug', 'fix', 'traceback', 'not working')):
        return 'debug_code'
    if any(w in t for w in ('explain', 'what does', 'how does', 'why')):
        return 'explain_code'
    return 'write_code'

---
**5. Generate with Groq**  
The retrieved chunks go into the prompt as reference material. The system prompt tells the model to prefer them (and follow the style guide) but to say so if they don't cover the request, rather than inventing APIs. Rate-limit errors are retried a few times with a short wait, since the free tier has per-minute caps.

In [47]:
import time

SYSTEM = '''You are CodeBud, a coding assistant.
You are given REFERENCE snippets retrieved from the user's knowledge base.
- Prefer the reference snippets and follow any style rules in them.
- If the references don't cover the request, answer from general knowledge and say so briefly.
- Never invent library functions that you are not sure exist.'''


def build_prompt(query, intent, hits):
    if hits:
        refs = '\n\n'.join(f'[{c["source"]}]\n{c["text"]}' for c, _ in hits)
    else:
        refs = '(no relevant references found)'
    return f'''REFERENCES:
{refs}

TASK TYPE: {intent}
INSTRUCTIONS: {INTENT_INSTRUCTIONS.get(intent, INTENT_INSTRUCTIONS['write_code'])}

USER REQUEST:
{query}'''


def codebud(query, k=3, show_sources=True, retries=3):
    intent = classify_intent(query)
    hits = retrieve(query, k=k)
    prompt = build_prompt(query, intent, hits)

    resp = None
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                max_tokens=1500,
                temperature=0.2,
                messages=[
                    {'role': 'system', 'content': SYSTEM},
                    {'role': 'user', 'content': prompt},
                ],
            )
            break
        except groq.RateLimitError:
            if attempt == retries - 1:
                return 'Rate limit hit. Wait a minute and try again.'
            time.sleep(5 * (attempt + 1))
        except groq.APIError as e:
            return f'API error: {e}'

    answer = resp.choices[0].message.content
    if show_sources:
        srcs = ', '.join(f'{c["source"]} ({s:.2f})' for c, s in hits) or 'none'
        print(f'[intent: {intent}] [sources: {srcs}]\n')
    return answer

In [48]:
print(codebud('Write a function that reads a JSON file and returns only the keys at the top level'))

[intent: write_code] [sources: MSE_regularization.ipynb (0.27), rfm.ipynb (0.25), Linear_Regression.ipynb (0.22)]

```python
import json
from typing import List

def get_top_level_keys(json_path: str) -> List[str]:
    """
    Read a JSON file and return a list containing only the keys that exist
    at the top level of the JSON object.

    Parameters
    ----------
    json_path: str
        Path to the JSON file to be read.

    Returns
    -------
    List[str]
        A list of top‑level keys. If the file does not contain a JSON object
        (e.g., it's a list or a primitive), an empty list is returned.
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # The JSON root must be a dict to have keys.
    if isinstance(data, dict):
        return list(data.keys())
    else:
        return []


# Example usage
if __name__ == "__main__":
    # Assume `sample.json` contains: {"name": "Alice", "age": 30, "city": "NY"}
    keys = get_top_level_

In [49]:
print(codebud('Write a function to check whether a list is sorted, in the CodeBud style'))

[intent: write_code] [sources: bigram.ipynb (0.24), Davidson_dataset_plot.ipynb (0.21), MSE_regularization.ipynb (0.21)]

```python
def is_sorted(lst: list) -> bool:
    """
    Check whether the given list is sorted in non‑decreasing (ascending) order.

    Parameters
    ----------
    lst : list
        The list to be examined.

    Returns
    -------
    bool
        True if the list is sorted, False otherwise.
    """
    # An empty list or a single‑element list is trivially sorted
    if len(lst) < 2:
        return True

    # Verify each adjacent pair respects the ordering
    return all(lst[i] <= lst[i + 1] for i in range(len(lst) - 1))


# Example usage
if __name__ == "__main__":
    test_cases = [
        [],                     # True
        [5],                    # True
        [1, 2, 3, 4, 5],        # True
        [1, 2, 2, 3],           # True (allows equal elements)
        [5, 4, 3, 2, 1],        # False
        [1, 3, 2, 4],           # False
    ]

    for case i

---
**Next steps**
- Drop your own code, docs and notes into `knowledge_base/` and re-run sections 2-3 (re-embedding is quick at this size).
- Replace `classify_intent` with a Module 1 model trained on coding intents.
- Cache `chunk_vecs` to disk with `np.save` once the knowledge base gets big.
- If the knowledge base grows to thousands of chunks, switch the dot product to FAISS or Chroma.